<a id="top"></a>

# QFT repeated-secret study

**Research question.** Can Eves pool sparse measurements over repeated sends of the same four-bit secret and recover it before Bob alarms?

**Start here.** Run this notebook from the repository root. The default run reads the [saved 1,210-setting sweep](results/qft_factorial_results.csv) and draws its results figure using pandas and Matplotlib. The optional checks and simulations also use NumPy and SciPy.

- [Read the saved results](#qft-dashboard)
- [Follow one exchange and the stopping rules](#qft-methods)
- [Run a small validation and demo](#qft-demo)
- [Reproduce the full sweep](#qft-batch)
- [Trace the methods to their sources](#references)

See the [BB84 detector comparison](BB84.ipynb) for pooled QBER, U2, and U4 and the [project README](README.md) for the overall study.

<a id="qft-dashboard"></a>

## Read the saved results

The reference run covers 2- and 4-qubit blocks, 11 fixed noise profiles, and 100 trials per setting. Bob checked for an alarm at 20, 50, and 100 retained check bits. An unresolved trial ended after 100 checks, equivalent to 25 attempts.

| Readout | What to look for |
| --- | --- |
| Left panel | How often noise alone triggered an alarm at each fixed noise profile. The pooled rate was **160 of 2,200 trials (7.27%)** against a 5% target. |
| Right panel | How often Bob alarmed before Eve declared recovery or the budget ended, averaged across grid settings at each \(K \times f\). \(K\) is the number of Eves and \(f\) is the interception probability per Eve per qubit on each leg. |
| Recovery | **Zero of 118,800 attacked trials** produced a full-key declaration at 99% confidence. The setting means omit bit-by-bit confidence trajectories, so this run cannot quantify partial recovery. |

Use the optional table selector for details by noise and block size (`"primary"`), by factor (`"factors"`), or for the highest average full-key guesses (`"leakage_watch"`). The [run notes](results/README.md) explain the CSV fields, costs, and limits.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from quantum_study.qft_results import build_qft_summary, plot_qft_summary


RESULTS_CSV = Path("results/qft_factorial_results.csv")
DETAIL_TABLE = None  # "false_alarm", "primary", "factors", or "leakage_watch"

qft_factorial_results = pd.read_csv(RESULTS_CSV)
qft_dashboard = build_qft_summary(
    qft_factorial_results, expected_settings=1_210, target_false_alarm=0.05,
)
display(qft_dashboard["audit"])

qft_dashboard_figure = plot_qft_summary(
    qft_dashboard,
    block_sizes=sorted(qft_factorial_results["block_qubits"].unique()),
    noise_order=qft_factorial_results["noise_profile"].unique(),
)
plt.show()

if DETAIL_TABLE is not None:
    display(qft_dashboard[DETAIL_TABLE].round(4))

<a id="qft-methods"></a>

## Follow one exchange

1. **Bob sends.** Bob prepares a block with fresh private phases and transmits it to Alice. Each leg applies Pauli \(X\) or \(Z\) noise with the configured probabilities. All qubits arrive.
2. **Alice encodes and returns.** Each attempt carries the same four secret bits across several blocks. Each block pairs its payload with equally many check bits; their positions and values are randomized. Bob removes his phases and applies the inverse QFT to decode.
3. **Eve updates.** Each Eve can measure and resend on either leg. The coalition combines its records across blocks and retransmissions, using the check disclosures and averaging over possible Bob phases to update probabilities for the 16 candidate secrets.
4. **Bob monitors.** Bob adds the negative log-likelihood of each observed check pattern under the attack-free model. Noise-only calibration accounts for growth across the alarm looks. This baseline records the first outcome among Bob's alarm, a 99%-confident full-key guess, and the final check budget. Bob wins a same-block tie.

The sweep calibrates each fixed noise profile separately. Testing channel drift would require scoring new noise-only trials against a frozen baseline. Measuring partial recovery would require saving per-bit posterior probabilities over attempts. The saved baseline answers the fixed-noise, full-key question described above.

**Source map.** The two-pass exchange and hidden checks follow Papadopoulos and Linvill; the coalition posterior, cumulative alarm, and factorial settings are project methods. See the [detailed source notes](#references) and [simulation implementation](quantum_study/qft_experiment.py).

<a id="qft-demo"></a>

## Run a small validation and demo

Set `RUN_MODEL_CHECKS = True` to verify all noiseless 2- and 4-qubit messages, compare scalar and batched likelihoods, and check recovery in a deliberately strong X-basis attack. Set `RUN_DEMO = True` to inspect eight example trials under the Erven reference profile. The demo illustrates individual stopping times and confidence; the saved sweep is the source for rate estimates.

`BACKEND = "auto"` probes CUDA and uses NumPy if the probe fails. Choose `"numpy"` for CPU or `"cupy"` to require the GPU. See the [GPU dependencies](requirements-gpu.txt) for the CUDA 13 setup.

In [ ]:
RUN_MODEL_CHECKS = False
RUN_DEMO = False
BACKEND = "auto"
GPU_PRECISION = "complex64"

if RUN_MODEL_CHECKS or RUN_DEMO:
    from quantum_study.qft_experiment import (
        configure_qft_backend, run_qft_demo, validate_qft_model,
    )

    backend = configure_qft_backend(BACKEND, gpu_precision=GPU_PRECISION)
    print(f"Posterior backend: {backend.name} ({backend.precision})")
    if RUN_MODEL_CHECKS:
        validation = validate_qft_model(backend=backend, verbose=True)
    if RUN_DEMO:
        qft_recovery_results, demo_summary, qft_alarm_rule = run_qft_demo(
            backend=backend, return_diagnostics=True,
        )
        display(qft_recovery_results)
        display(demo_summary)

<a id="qft-batch"></a>

## Reproduce the full sweep

This implementation combines the proposal's factor levels into 1,210 settings using a common interception rate for all Eves. The saved run used 100 trials per setting and took about 131 minutes on the RTX 5090. Set `RUN_FULL_QFT_SWEEP = True` to reproduce this baseline. NumPy and CuPy use the same posterior calculation.

The default checkpoint path can resume the original run when the settings, backend, and precision match. For a fresh run, use a new checkpoint path. The completed output goes to a separate reproduction CSV. To review it, point `RESULTS_CSV` at that file and rerun the results cell.

In [ ]:
RUN_FULL_QFT_SWEEP = False
RUNS_PER_SETTING = 100
PHASE_SAMPLES = 16
CALIBRATION_RUNS = 100
BATCH_SIZE = 128
RESUME_SWEEP = True
CHECKPOINT_PATH = Path("results/qft_factorial_checkpoint.json")
SWEEP_RESULTS_CSV = Path("results/qft_factorial_reproduction.csv")

if RUN_FULL_QFT_SWEEP:
    from quantum_study.qft_experiment import (
        configure_qft_backend, run_qft_factorial_sweep,
    )

    backend = configure_qft_backend(BACKEND, gpu_precision=GPU_PRECISION)
    reproduced_results = run_qft_factorial_sweep(
        runs_per_setting=RUNS_PER_SETTING,
        checkpoint_path=CHECKPOINT_PATH,
        results_path=SWEEP_RESULTS_CSV,
        resume=RESUME_SWEEP,
        backend=backend,
        phase_samples=PHASE_SAMPLES,
        calibration_runs=CALIBRATION_RUNS,
        batch_size=BATCH_SIZE,
    )
    print(f"Saved {len(reproduced_results):,} settings to {SWEEP_RESULTS_CSV}")

<a id="references"></a>

## Methods and references

- **Papadopoulos and Linvill (2025), _Increasing Interference Detection in Quantum Cryptography Using the Quantum Fourier Transform_.** The scrambler, phase encoding, and inverse-QFT decoding correspond to equations (1)–(3) in §4.1 and Fig. 2. Interference propagation follows §4.2 and §§5.1–5.3, especially equations (7)–(11). The repeated-copy motivation and hidden checks follow §§3.2 and 6.1. [arXiv:2404.12507](https://arxiv.org/abs/2404.12507)
- **Dehmani et al. (2012), _Quantum Key Distribution with Several Intercept-Resend Attacks via a Depolarizing Channel_.** Motivates studying multiple sequential Eves. [arXiv:1302.7206](https://arxiv.org/abs/1302.7206)
- **Erven et al. (2009), _Entangled Quantum Key Distribution with a Biased Basis Choice_.** Supplies the reference noise-profile values used in the demo and sweep. [arXiv:0901.0960](https://arxiv.org/abs/0901.0960)
- **Napieraj and Chuang (2026), _Detecting Patient Attacks on a Repeated Secret_, §2.** The study proposal specifies the block sizes, channel and attack factors, joint check-pattern likelihoods, the 99% recovery outcome, and a 100-attempt cap. Phase sampling and the same-block timing rule are implementation choices. The saved run covers 25 attempts.

[Back to top](#top)